# Random Walk

Deep-dive notebook companion to the [Random Walk wiki page](https://ml-viz-ruby.vercel.app/wiki/random-walk).

**What you'll build here:**
- Simulate 1D and 2D random walks and verify theoretical mean/variance
- Visualise how variance grows linearly with time
- Derive and check the Gambler's Ruin formula
- Show the Brownian-motion limit via the CLT
- Implement a bare-bones Metropolis-Hastings sampler using a random-walk proposal

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor':  '#0f1117',
    'axes.facecolor':    '#1a1d27',
    'axes.edgecolor':    '#2d3148',
    'axes.labelcolor':   '#e2e8f0',
    'xtick.color':       '#94a3b8',
    'ytick.color':       '#94a3b8',
    'text.color':        '#e2e8f0',
    'grid.color':        '#2d3148',
    'lines.linewidth':   1.5,
    'font.size':         11,
})
BRAND   = '#6366f1'   # indigo
TEAL    = '#2dd4bf'
ROSE    = '#fb7185'
YELLOW  = '#fbbf24'
MUTED   = '#475569'

rng = np.random.default_rng(42)

## 1 · Definition and basic simulation

A **random walk** is defined by accumulating i.i.d. increments:

$$S_t = \sum_{i=1}^{t} \varepsilon_i, \qquad S_0 = 0$$

For the symmetric case, $\varepsilon_i \in \{-1, +1\}$ with equal probability.

**Key properties:**
- $\mathrm{E}[S_t] = 0$ (unbiased)
- $\mathrm{Var}(S_t) = t\sigma^2$ (grows linearly with time)
- Markov property: $S_{t+1} = S_t + \varepsilon_{t+1}$ depends only on the current position

In [ ]:
def simulate_walks(n_steps: int, n_paths: int = 1, sigma: float = 1.0) -> np.ndarray:
    """Return array of shape (n_paths, n_steps+1) with each row a random walk path."""
    increments = rng.normal(0.0, sigma, size=(n_paths, n_steps))
    paths = np.concatenate([np.zeros((n_paths, 1)), np.cumsum(increments, axis=1)], axis=1)
    return paths

T = 300
paths = simulate_walks(T, n_paths=40)
t     = np.arange(T + 1)

fig, ax = plt.subplots(figsize=(10, 4))
for path in paths:
    ax.plot(t, path, color=BRAND, alpha=0.25, lw=0.8)
ax.fill_between(t, -np.sqrt(t), np.sqrt(t), alpha=0.15, color=TEAL, label=r'$\pm\sigma\sqrt{t}$ band')
ax.axhline(0, color=MUTED, lw=0.8, ls='--')
ax.set_xlabel('Step $t$')
ax.set_ylabel('$S_t$')
ax.set_title('40 independent random walk paths with theoretical ±1 SD band')
ax.legend()
plt.tight_layout()
plt.show()

## 2 · Verifying the variance grows linearly

Theory says $\mathrm{Var}(S_t) = t$. Let's verify empirically across several checkpoints.

In [ ]:
T_long  = 1000
N_paths = 5000
big_walks = simulate_walks(T_long, n_paths=N_paths)

checkpoints = [1, 5, 10, 50, 100, 250, 500, 1000]
emp_var     = [big_walks[:, t_].var() for t_ in checkpoints]

print(f"{'t':>6}  {'Empirical Var':>14}  {'Theoretical t':>14}  {'Ratio':>8}")
print("-" * 48)
for t_, ev in zip(checkpoints, emp_var):
    print(f"{t_:>6}  {ev:>14.3f}  {float(t_):>14.1f}  {ev/t_:>8.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Left: variance vs time
all_t  = np.arange(1, T_long + 1)
emp_all = big_walks[:, 1:].var(axis=0)
axes[0].plot(all_t, emp_all, color=BRAND, lw=1, label='Empirical $\\mathrm{Var}(S_t)$')
axes[0].plot(all_t, all_t,   color=TEAL,  lw=1.5, ls='--', label='Theoretical $t$')
axes[0].set(xlabel='$t$', ylabel='Variance', title='Variance grows as $t$')
axes[0].legend()

# Right: distribution at different times
for t_, col in [(50, BRAND), (200, TEAL), (800, ROSE)]:
    axes[1].hist(big_walks[:, t_], bins=60, density=True, alpha=0.45,
                 color=col, label=f'$t={t_}$')
axes[1].set(xlabel='$S_t$', ylabel='Density', title='Distribution at different times')
axes[1].legend()

plt.tight_layout()
plt.show()

## 3 · The Brownian-motion limit

Scale a symmetric $\{\pm 1\}$ walk so that each step covers $\Delta x = 1/\sqrt{n}$ in
time $\Delta t = 1/n$. As $n \to \infty$, the process on $[0, 1]$ converges to
standard Brownian motion $W_t \sim \mathcal{N}(0, t)$.

Below we show that for increasing resolution the empirical distribution at $t = 1$
converges to $\mathcal{N}(0, 1)$.

In [ ]:
from scipy.stats import norm

fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=False)
ns = [10, 100, 1000]

for ax, n in zip(axes, ns):
    # n steps of ±1/sqrt(n) → final position is S_n / sqrt(n)
    raw_walks = rng.choice([-1, 1], size=(10_000, n)).cumsum(axis=1)
    final     = raw_walks[:, -1] / np.sqrt(n)

    ax.hist(final, bins=60, density=True, color=BRAND, alpha=0.7, label=f'$n={n}$')
    xs = np.linspace(-4, 4, 300)
    ax.plot(xs, norm.pdf(xs), color=TEAL, lw=2, label='$\\mathcal{N}(0,1)$')
    ax.set(xlabel='$S_n / \\sqrt{n}$', title=f'$n = {n}$ steps')
    ax.legend(fontsize=9)

fig.suptitle("Convergence to Brownian motion (CLT)", y=1.02)
plt.tight_layout()
plt.show()

## 4 · Gambler's Ruin

A gambler starts at position $k$, wins $+1$ with probability $p$, loses $-1$ with
probability $q = 1 - p$, and stops at $0$ (ruined) or $N$ (wins). The ruin probability
is:

$$
R_k = \begin{cases}
\dfrac{(q/p)^k - (q/p)^N}{1 - (q/p)^N} & p \ne \tfrac{1}{2} \\[8pt]
1 - k/N & p = \tfrac{1}{2}
\end{cases}
$$

In [ ]:
def ruin_prob_analytical(k: int, N: int, p: float) -> float:
    q = 1.0 - p
    if abs(p - 0.5) < 1e-12:
        return 1.0 - k / N
    r = q / p
    return (r**k - r**N) / (1.0 - r**N)

def ruin_prob_simulation(k: int, N: int, p: float, n_sims: int = 100_000) -> float:
    """Estimate ruin probability by Monte Carlo."""
    positions = np.full(n_sims, float(k))
    active    = np.ones(n_sims, dtype=bool)
    ruined    = np.zeros(n_sims, dtype=bool)
    while active.any():
        steps = rng.choice([-1, 1], size=n_sims, p=[q := 1 - p, p])
        positions[active] += steps[active]
        ruined[active]  |= positions[active] <= 0
        active &= (positions > 0) & (positions < N)
    return ruined.mean()

N = 50
print(f"Gambler's Ruin — N={N}, comparison of analytical vs simulation")
print(f"{'k':>4}  {'p':>5}  {'Analytical':>12}  {'Simulation':>12}")
print("-" * 40)
for k, p in [(5, 0.50), (10, 0.50), (5, 0.48), (25, 0.49)]:
    ana = ruin_prob_analytical(k, N, p)
    sim = ruin_prob_simulation(k, N, p, n_sims=50_000)
    print(f"{k:>4}  {p:>5.2f}  {ana:>12.4f}  {sim:>12.4f}")

In [ ]:
# Ruin probability as a function of starting position for several p values
N = 100
ks = np.arange(1, N)

fig, ax = plt.subplots(figsize=(9, 4))
for p, col, label in [(0.51, ROSE, '$p=0.51$ (house edge)'),
                       (0.50, BRAND, '$p=0.50$ (fair)'),
                       (0.49, TEAL, '$p=0.49$ (player edge)')]:
    ax.plot(ks, [ruin_prob_analytical(k, N, p) for k in ks], color=col, label=label)

ax.set(xlabel='Starting chips $k$', ylabel='Ruin probability $R_k$',
       title=f"Gambler's Ruin — target $N={N}$ chips")
ax.legend()
plt.tight_layout()
plt.show()

## 5 · Random walk in 2D

In 2D the walk moves one unit along a randomly chosen axis direction at each step.
Pólya's theorem guarantees recurrence: the walk returns to the origin with probability 1.

In [ ]:
def random_walk_2d(n_steps: int) -> np.ndarray:
    dirs = rng.choice(
        [[1, 0], [-1, 0], [0, 1], [0, -1]],
        size=n_steps
    )
    return np.vstack([np.zeros((1, 2)), np.cumsum(dirs, axis=0)])

path = random_walk_2d(20_000)
distances = np.linalg.norm(path, axis=1)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: the 2D path
n = len(path)
colors = plt.cm.plasma(np.linspace(0, 1, n))
for i in range(n - 1):
    axes[0].plot(path[i:i+2, 0], path[i:i+2, 1], color=colors[i], lw=0.4, alpha=0.6)
axes[0].scatter(*path[0],  color=TEAL,  s=60, zorder=5, label='Start')
axes[0].scatter(*path[-1], color=ROSE,  s=60, zorder=5, label='End')
axes[0].set(xlabel='$x$', ylabel='$y$', title='2D random walk (20 000 steps)', aspect='equal')
axes[0].legend()

# Right: distance from origin
t_arr = np.arange(len(path))
axes[1].plot(t_arr, distances, color=BRAND, lw=0.7, alpha=0.8, label='$\\|S_t\\|$')
axes[1].plot(t_arr, np.sqrt(t_arr), color=TEAL, lw=1.5, ls='--', label='$\\sqrt{t}$ (expected SD)')
axes[1].set(xlabel='Step $t$', ylabel='Distance from origin',
            title='Distance from origin over time')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Final position: ({path[-1,0]:.0f}, {path[-1,1]:.0f})")
print(f"Expected ‖S_T‖ ~ √T = {np.sqrt(len(path)):.1f}")

## 6 · Random-walk Metropolis-Hastings

The Metropolis-Hastings algorithm uses a random-walk proposal to sample from an
arbitrary target density $p(x)$:

1. Propose $x' = x + \delta$, $\delta \sim \mathcal{N}(0, \sigma^2)$
2. Accept with probability $\min(1,\ p(x') / p(x))$
3. If accepted, move to $x'$; otherwise stay at $x$

Below we sample from a bimodal mixture target.

In [ ]:
from scipy.stats import norm as sp_norm

def target_log_prob(x: float) -> float:
    """Log-density of a bimodal mixture: 0.5 N(-3,1) + 0.5 N(3,1)."""
    return np.log(0.5 * sp_norm.pdf(x, -3, 1) + 0.5 * sp_norm.pdf(x, 3, 1))

def metropolis_hastings(log_target, n_samples: int = 50_000,
                         sigma: float = 1.0, x0: float = 0.0):
    samples = np.empty(n_samples)
    x       = x0
    n_accept = 0
    for i in range(n_samples):
        x_prop  = x + rng.normal(0, sigma)
        log_a   = log_target(x_prop) - log_target(x)
        if np.log(rng.uniform()) < log_a:
            x        = x_prop
            n_accept += 1
        samples[i] = x
    return samples, n_accept / n_samples

# Compare step sizes
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
xs = np.linspace(-8, 8, 500)
target_density = np.exp([target_log_prob(x) for x in xs])

for ax, sigma in zip(axes, [0.1, 1.0, 5.0]):
    samples, rate = metropolis_hastings(target_log_prob, n_samples=30_000, sigma=sigma)
    burn_in       = 2_000
    ax.hist(samples[burn_in:], bins=80, density=True, color=BRAND,
            alpha=0.65, label=f'MH samples (accept={rate:.2f})')
    ax.plot(xs, target_density, color=TEAL, lw=2, label='True density')
    ax.set(xlabel='$x$', title=f'Step size $\\sigma = {sigma}$')
    ax.legend(fontsize=9)

fig.suptitle('MH with random-walk proposal — effect of step size on mixing', y=1.02)
plt.tight_layout()
plt.show()

## ✏️ Your turn

Three exercises to reinforce the key ideas.

---

### Exercise 1 — Reflection principle

**Concept:** The reflection principle says:
$$P\!\left(\max_{0 \le s \le T} S_s \ge a\right) = 2\,P(S_T \ge a)$$
for a symmetric Gaussian random walk and $a > 0$.

**Task:** For $T = 100$ steps (unit-variance increments) and $a = 5$:
1. Compute the theoretical probability using the reflection principle
2. Verify it by Monte Carlo simulation

In [ ]:
T_ex1 = 100
a_ex1 = 5.0

# --- Theoretical probability (reflection principle) ---
# P(max S_s >= a) = 2 * P(S_T >= a)
# S_T ~ N(0, T), so P(S_T >= a) = 1 - Phi(a / sqrt(T))
from scipy.stats import norm as sp_norm2

# TODO(you): compute p_theoretical using the reflection principle formula
p_theoretical = None  # replace with your computation

# --- Simulation ---
n_sims_ex1 = 100_000

# TODO(you): simulate n_sims_ex1 random walks of T_ex1 steps,
# compute the fraction that hit level a_ex1 at some point
p_simulation = None  # replace with your computation

print(f"Theoretical (reflection principle): {p_theoretical}")
print(f"Simulation:                         {p_simulation}")

In [ ]:
# --- Assertion cell (runs silently when correct) ---
assert p_theoretical is not None, "Compute p_theoretical first"
assert p_simulation  is not None, "Compute p_simulation first"
assert abs(p_theoretical - 2 * (1 - sp_norm2.cdf(a_ex1 / np.sqrt(T_ex1)))) < 1e-9, \
    "p_theoretical doesn't match the reflection-principle formula"
assert abs(p_simulation - p_theoretical) < 0.02, \
    f"Simulation ({p_simulation:.4f}) diverges from theory ({p_theoretical:.4f}) by more than 2pp"
print("✓ Exercise 1 passed")

<details>
<summary>Solution — Exercise 1</summary>

```python
# Theoretical
p_theoretical = 2 * (1 - sp_norm2.cdf(a_ex1 / np.sqrt(T_ex1)))

# Simulation
walks_ex1  = simulate_walks(T_ex1, n_paths=n_sims_ex1)  # shape (n_sims, T+1)
max_values = walks_ex1.max(axis=1)
p_simulation = (max_values >= a_ex1).mean()
```

With $T=100$ and $a=5$: $P(S_{100} \ge 5) = 1 - \Phi(5/10) = 1 - \Phi(0.5) \approx 0.3085$,
so the reflection principle gives $\approx 0.617$.
</details>

---

### Exercise 2 — Non-stationarity via variance test

**Concept:** A stationary process has constant variance. A random walk has $\mathrm{Var}(S_t) = t$.

**Task:** Given two mystery series `series_a` and `series_b` (one stationary AR(1), one random walk),
compute the empirical variance at $t = 10$, $t = 100$, and $t = 500$. Identify which is which.

In [ ]:
# Generate mystery series (don't peek at the labels yet)
_phi        = 0.92
_n_paths    = 3000
_T          = 600
_ar1_noise  = rng.normal(0, 1, size=(_n_paths, _T))
_ar1_paths  = np.zeros((_n_paths, _T + 1))
for _t in range(1, _T + 1):
    _ar1_paths[:, _t] = _phi * _ar1_paths[:, _t-1] + _ar1_noise[:, _t-1]

_rw_paths = simulate_walks(_T, n_paths=_n_paths)

# Shuffle so you don't know which is which
_mystery_order = rng.permutation(2)
_all_series    = [_ar1_paths, _rw_paths]
series_a = _all_series[_mystery_order[0]]
series_b = _all_series[_mystery_order[1]]

# TODO(you): compute variance of series_a and series_b at t = 10, 100, 500
# Then decide: which series is the random walk?
# Assign your answer to `rw_is` as either 'a' or 'b'
rw_is = None  # 'a' or 'b'

In [ ]:
assert rw_is in ('a', 'b'), "Set rw_is to 'a' or 'b'"
correct_label = 'a' if _mystery_order[0] == 1 else 'b'
assert rw_is == correct_label, \
    f"Incorrect — the random walk was series_{correct_label}. " \
    f"Hint: its variance grows like t (10 → 100 → 500), the AR(1)'s stays bounded."
print("✓ Exercise 2 passed — you correctly identified the random walk")

<details>
<summary>Solution — Exercise 2</summary>

```python
for name, s in [('a', series_a), ('b', series_b)]:
    for t_ in [10, 100, 500]:
        print(f"series_{name}  t={t_:>3}  Var = {s[:, t_].var():.2f}")
```

The random walk will show variance ≈ 10, ≈ 100, ≈ 500 (tracking $t$ closely).
The stationary AR(1) with $|\phi| < 1$ will show variance that plateaus near
$\sigma^2 / (1 - \phi^2)$ regardless of $t$.
</details>

---

### Exercise 3 — Tuning the Metropolis-Hastings step size

**Concept:** The mixing quality of MH with a random-walk proposal depends critically
on the step size $\sigma$. The Goldilocks target for a univariate Gaussian target is
an acceptance rate near **0.44** (Roberts & Rosenthal, 2001).

**Task:** Given the target `target_log_prob` defined earlier (bimodal mixture of
two Gaussians centred at ±3 with unit variance), find the step size $\sigma^*$
such that the acceptance rate is closest to 0.44. Search over the grid
`[0.2, 0.5, 1.0, 1.5, 2.0, 3.0]` and store the best $\sigma$ in `best_sigma`.

In [ ]:
sigma_grid = [0.2, 0.5, 1.0, 1.5, 2.0, 3.0]
target_rate = 0.44

# TODO(you): iterate over sigma_grid, run metropolis_hastings for each,
# and find the sigma with acceptance rate closest to target_rate.
# Store it in best_sigma.
best_sigma = None

In [ ]:
assert best_sigma is not None, "Compute best_sigma"
_, best_rate = metropolis_hastings(target_log_prob, n_samples=30_000, sigma=best_sigma)
assert abs(best_rate - target_rate) < 0.15, \
    f"Acceptance rate {best_rate:.3f} for sigma={best_sigma} is too far from 0.44"
print(f"✓ Exercise 3 passed — best sigma = {best_sigma}, acceptance rate ≈ {best_rate:.3f}")

<details>
<summary>Solution — Exercise 3</summary>

```python
rates = {}
for sigma in sigma_grid:
    _, rate = metropolis_hastings(target_log_prob, n_samples=20_000, sigma=sigma)
    rates[sigma] = rate
    print(f"sigma={sigma:.1f}  accept={rate:.3f}")

best_sigma = min(rates, key=lambda s: abs(rates[s] - target_rate))
print(f"\nBest sigma: {best_sigma}  (rate ≈ {rates[best_sigma]:.3f})")
```

For the $0.5 \mathcal{N}(-3,1) + 0.5 \mathcal{N}(3,1)$ target, $\sigma \approx 1.0$–$1.5$ typically
lands nearest 0.44. Larger $\sigma$ values mix between modes faster but have lower
acceptance rates; smaller $\sigma$ values accept almost everything but explore slowly.
</details>